In [ ]:
# ── CELL 1: MOUNT & INSTALL ────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
# !pip install statsmodels -q
#!pip install regex
#import regex as re

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import requests
import re
from io import StringIO
from pathlib import Path
from bs4 import BeautifulSoup
import time
import warnings
import os
os.remove('sep_dots_raw.csv')   # delete the old broken cache
warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════

MASTER_CSV_PATH = "/content/drive/MyDrive/HEC Thesis/Data/Master_Macro.csv"
OUTPUT_PATH     = "/content/drive/MyDrive/HEC Thesis/Data/Master_Macro_with_SEP.csv"
MASTER_DATE_COL = "date"

SEP_CACHE_CSV   = "sep_dots_raw.csv"   # won't re-scrape if this exists
FILL_METHOD     = 0                    # 0 = zero-fill non-SEP meetings

# ══════════════════════════════════════════════════════════════════════════════
# STEP 1 — LOAD MASTER
# ══════════════════════════════════════════════════════════════════════════════

print("=== Loading Master_Macro.csv ===")
master = pd.read_csv(MASTER_CSV_PATH)

# Strip whitespace from all column names — hidden spaces are a common cause
# of KeyError even when the column looks correct in a spreadsheet viewer
master.columns = master.columns.str.strip()

# Print first few columns so you can see exactly what came in
print(f"  Columns (first 10): {list(master.columns[:10])}")

# Auto-detect date column: try MASTER_DATE_COL first, then common alternatives,
# then any column that parses cleanly as dates
date_col = None
candidates = [MASTER_DATE_COL, 'meeting_date', 'Date', 'DATE',
              'Meeting_Date', 'observation_date']
for c in candidates:
    if c in master.columns:
        date_col = c
        break

if date_col is None:
    # Try any column that looks like dates
    for c in master.columns:
        try:
            parsed = pd.to_datetime(master[c], errors='coerce')
            if parsed.notna().mean() > 0.8:
                date_col = c
                print(f"  Auto-detected date column: '{c}'")
                break
        except Exception:
            pass

if date_col is None:
    # Last resort: check if the index is already datetime
    try:
        test = pd.to_datetime(master.index, errors='coerce')
        if test.notna().mean() > 0.8:
            master.index = pd.to_datetime(master.index)
            master.index.name = 'meeting_date'
            date_col = '__index__'
            print("  Date is already in the index.")
    except Exception:
        pass

if date_col is None:
    raise ValueError(
        f"Cannot find a date column. "
        f"Available columns: {list(master.columns)}\n"
        f"Set MASTER_DATE_COL at the top of the script to the correct name."
    )

if date_col != '__index__':
    master[date_col] = pd.to_datetime(master[date_col])
    master = master.set_index(date_col)

master = master.sort_index()
master.index.name = 'meeting_date'
print(f"  {len(master)} rows  |  "
      f"{master.index[0].date()} -> {master.index[-1].date()}\n")

# ══════════════════════════════════════════════════════════════════════════════
# STEP 2 — FETCH FRED SEP MEDIANS (fixed CSV parsing)
# ══════════════════════════════════════════════════════════════════════════════

def fetch_fred(series_id):
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
    try:
        r = requests.get(url, timeout=20)
        if r.status_code != 200:
            print(f"  X {series_id}: HTTP {r.status_code}")
            return None
        df = pd.read_csv(StringIO(r.text))
        df.columns = ['date', 'value']
        df['date']  = pd.to_datetime(df['date'], errors='coerce')
        df['value'] = pd.to_numeric(df['value'], errors='coerce')
        df = df.dropna(subset=['date']).set_index('date')
        df.columns = [series_id]
        print(f"  OK {series_id}: {len(df)} obs  "
              f"({df.index[0].date()} -> {df.index[-1].date()})")
        return df
    except Exception as e:
        print(f"  X {series_id}: {e}")
        return None


print("=== Fetching SEP medians from FRED ===\n")

# FRED restructured the FEDTARMD series in 2024 — it now only holds
# forward projections (2026+). The correct historical series are:
#   FEDTARMDLR  = longer-run median FFR projection  (2012–present) ✓
#   GDPC1MD     = median GDP growth projection       (forward-looking only)
#   UNRATEMED   = median unemployment projection     (forward-looking only)
#   PCECTPIMED  = median PCE inflation projection    (forward-looking only)
#
# For the year-specific medians (current year, +1, +2) FRED does not maintain
# clean historical time series — those require scraping the Fed's HTML tables.
# We fetch what FRED reliably has and supplement with the scraper below.
FRED_SERIES = {
    'sep_ffr_med_lr':   'FEDTARMDLR',   # longer-run FFR median  2012–present
    'sep_ffr_range_lo': 'FEDTARRLLO',   # range low  (for dispersion proxy)
    'sep_ffr_range_hi': 'FEDTARRRHI',   # range high (for dispersion proxy)
    'sep_ffr_ct_lo':    'FEDTARCTL',    # central tendency low
    'sep_ffr_ct_hi':    'FEDTARCTH',    # central tendency high
    'sep_ffr_ct_mid':   'FEDTARCTM',    # central tendency midpoint
}

fred_frames = []
for col, sid in FRED_SERIES.items():
    df_s = fetch_fred(sid)
    if df_s is not None:
        df_s.columns = [col]
        fred_frames.append(df_s)

# Derive a crude dispersion measure from range hi - range lo if available
df_fred = pd.concat(fred_frames, axis=1) if fred_frames else pd.DataFrame()
if 'sep_ffr_range_lo' in df_fred.columns and 'sep_ffr_range_hi' in df_fred.columns:
    df_fred['sep_ffr_range_width'] = (df_fred['sep_ffr_range_hi']
                                      - df_fred['sep_ffr_range_lo'])
if 'sep_ffr_ct_lo' in df_fred.columns and 'sep_ffr_ct_hi' in df_fred.columns:
    df_fred['sep_ffr_ct_width'] = (df_fred['sep_ffr_ct_hi']
                                   - df_fred['sep_ffr_ct_lo'])
print(f"\n  Fetched {len(df_fred.columns)} FRED series.\n")

# ══════════════════════════════════════════════════════════════════════════════
# STEP 3 — SCRAPE FULL DOT DISTRIBUTION FROM THE FED WEBSITE
# ══════════════════════════════════════════════════════════════════════════════

# All SEP meeting dates (dot plot introduced Jan 2012)
SEP_MEETING_DATES = [
    "20120125","20120425","20120620","20120913","20121212",
    "20130320","20130619","20130918","20131218",
    "20140319","20140618","20140917","20141217",
    "20150318","20150617","20150917","20151216",
    "20160316","20160615","20160921","20161214",
    "20170315","20170614","20170920","20171213",
    "20180321","20180613","20180926","20181219",
    "20190320","20190619","20190918","20191211",
    "20200129","20200318","20200610","20200916","20201216",
    "20210317","20210616","20210922","20211215",
    "20220316","20220615","20220921","20221214",
    "20230201","20230322","20230614","20230920","20231213",
    "20240320","20240612","20240918","20241218",
    "20250319","20250618",
]


def parse_sep_table(html, meeting_date_str):
    """
    Parse the Fed SEP HTML page. Handles two distinct table formats:

    FORMAT A (2012–mid 2014): count-per-bucket table
      Rows = FFR level buckets  |  Cols = year projections
      Cell value = number of participants choosing that level

    FORMAT B (late 2014–present): direct dot-value table
      The Fed switched to a layout where participant counts per bucket
      are shown differently, or the table may now be embedded as an
      object/PDF link. We try multiple parsing strategies.
    """
    soup  = BeautifulSoup(html, 'lxml')
    stats = {'meeting_date': pd.to_datetime(meeting_date_str, format='%Y%m%d')}

    # ── Strategy 1: look for any table containing FFR level data ─────────────
    tables = soup.find_all('table')

    for table in tables:
        text = table.get_text(' ', strip=True).lower()
        # Must contain some rate-related content
        if not any(kw in text for kw in
                   ['federal funds', 'funds rate', 'ffr', '0 to', '0.25 to']):
            continue

        rows      = table.find_all('tr')
        col_years = []
        levels    = []
        counts    = {}

        for row in rows:
            cells = [c.get_text(strip=True) for c in row.find_all(['td','th'])]
            if not cells:
                continue

            cell_text = ' '.join(cells)

            # ── Header row detection: contains year numbers ───────────────
            year_found = [str(y) for y in range(2012, 2031) if str(y) in cell_text]
            lr_found   = any(w in cell_text.lower()
                             for w in ['longer run','longer-run','long run','lr'])
            if year_found or lr_found:
                col_years = []
                for cell in cells[1:]:
                    matched_yr = None
                    for y in range(2012, 2031):
                        if str(y) in cell:
                            matched_yr = str(y)
                            break
                    if matched_yr:
                        col_years.append(matched_yr)
                    elif any(w in cell.lower()
                             for w in ['longer run','longer-run','long run','lr']):
                        col_years.append('lr')
                continue

            if not col_years:
                continue

            # ── Data row: parse FFR midpoint from label ───────────────────
            label = cells[0]
            nums  = []

            # Handle formats: "0 to 1/4", "4-1/4 to 4-1/2", "5.25 to 5.50"
            # First normalise fractions written as X-Y/Z (e.g. "4-1/4")
            # Convert "N-P/Q" to "N+P/Q"
            label_norm = re.sub(r'(\d)-(\d)/(\d)',
                                lambda m: str(int(m.group(1)) + int(m.group(2))/int(m.group(3))),
                                label)
            for part in re.split(r'[to\s\-–]+', label_norm):
                part = part.strip()
                if not part:
                    continue
                try:
                    if '/' in part:
                        n, d = part.split('/')
                        nums.append(float(n)/float(d))
                    else:
                        nums.append(float(part))
                except ValueError:
                    pass

            if not nums:
                continue

            midpoint = sum(nums) / len(nums)
            levels.append(midpoint)

            for i, yr in enumerate(col_years):
                if i + 1 < len(cells):
                    try:
                        count = int(cells[i+1])
                    except ValueError:
                        count = 0
                    counts.setdefault(yr, []).append(count)

        # Compute statistics from whatever we parsed
        for yr, cnt_list in counts.items():
            if len(cnt_list) != len(levels) or sum(cnt_list) == 0:
                continue

            obs = []
            for level, count in zip(levels, cnt_list):
                obs.extend([level] * count)
            obs.sort()
            if not obs:
                continue

            med    = float(np.median(obs))
            p25    = float(np.percentile(obs, 25))
            p75    = float(np.percentile(obs, 75))
            mn     = float(obs[0])
            mx     = float(obs[-1])
            iqr    = p75 - p25
            skew_b = (p75 + p25 - 2*med) / iqr if iqr > 0 else 0.0
            tail_a = (mx - med) - (med - mn)

            suffix = yr if yr == 'lr' else yr[-2:]

            stats[f'sep_ffr_median_{suffix}']      = med
            stats[f'sep_ffr_p25_{suffix}']         = p25
            stats[f'sep_ffr_p75_{suffix}']         = p75
            stats[f'sep_ffr_min_{suffix}']         = mn
            stats[f'sep_ffr_max_{suffix}']         = mx
            stats[f'sep_ffr_iqr_{suffix}']         = iqr
            stats[f'sep_ffr_skew_bowley_{suffix}'] = skew_b
            stats[f'sep_ffr_tail_asym_{suffix}']   = tail_a
            stats[f'sep_ffr_n_{suffix}']           = len(obs)

    # ── Strategy 2: try the PDF URL if HTML gave nothing ─────────────────────
    # From ~2015 onward the Fed publishes the dot table as a PDF companion.
    # We cannot parse the PDF here directly — flag for manual review.
    if len(stats) <= 1:
        stats['_parse_failed'] = True

    return stats if len([k for k in stats if not k.startswith('_')]) > 1 else None


def scrape_all_sep(meeting_dates, cache_path, pause=1.2):
    cache = Path(cache_path)
    if cache.exists():
        print(f"  Loading from cache: {cache_path}")
        print(f"  (Delete '{cache_path}' and re-run to force a fresh scrape)")
        df = pd.read_csv(cache_path, parse_dates=['meeting_date'],
                         index_col='meeting_date')
        print(f"  {len(df)} meetings loaded from cache.\n")
        return df

    print(f"  Scraping {len(meeting_dates)} pages — this takes ~2 mins...\n")
    records  = []
    base_url = "https://www.federalreserve.gov/monetarypolicy/fomcprojtabl{}.htm"

    for i, d in enumerate(meeting_dates):
        url = base_url.format(d)
        try:
            r = requests.get(url, timeout=20,
                             headers={'User-Agent': 'Mozilla/5.0 (research)'})
            if r.status_code == 200:
                result = parse_sep_table(r.text, d)
                if result:
                    records.append(result)
                    print(f"  [{i+1:3d}/{len(meeting_dates)}] OK {d}  "
                          f"({len(result)-1} stats)")
                else:
                    print(f"  [{i+1:3d}/{len(meeting_dates)}] X  {d}  (parse failed)")
            else:
                print(f"  [{i+1:3d}/{len(meeting_dates)}] X  {d}  HTTP {r.status_code}")
        except Exception as e:
            print(f"  [{i+1:3d}/{len(meeting_dates)}] X  {d}  {e}")
        time.sleep(pause)

    if not records:
        print("  No records scraped.\n")
        return pd.DataFrame()

    df = pd.DataFrame(records).set_index('meeting_date').sort_index()
    df.to_csv(cache_path)
    print(f"\n  Scraped {len(df)} meetings -> cached to '{cache_path}'\n")
    return df


print("=== Scraping SEP dot distribution from federalreserve.gov ===\n")
df_dots = scrape_all_sep(SEP_MEETING_DATES, SEP_CACHE_CSV)

# ══════════════════════════════════════════════════════════════════════════════
# STEP 4 — COMPUTE REVISION MEASURES
# ══════════════════════════════════════════════════════════════════════════════

if not df_dots.empty:
    print("=== Computing revision measures ===\n")
    for col in [c for c in df_dots.columns if 'median' in c]:
        df_dots[col.replace('median','revision')]     = df_dots[col].diff()
        df_dots[col.replace('median','revision_abs')] = df_dots[col].diff().abs()
    print(f"  Revision columns added.\n")

# ══════════════════════════════════════════════════════════════════════════════
# STEP 5 — ALIGN TO MASTER MEETING DATES
# ══════════════════════════════════════════════════════════════════════════════

TOLERANCE_DAYS = 5

def align_to_master(master_df, source_df, fill=0, tol=5, label=''):
    if source_df.empty:
        print(f"  {label}: empty — skipping")
        return master_df

    new_cols = [c for c in source_df.columns if c not in master_df.columns]
    fill_val = np.nan if fill == 'ffill' else \
               (np.nan if (isinstance(fill, float) and np.isnan(fill)) else float(fill))
    for col in new_cols:
        master_df[col] = fill_val

    matched = unmatched = 0
    for src_date in source_df.index:
        diffs = abs(master_df.index - src_date)
        if diffs.min() <= pd.Timedelta(days=tol):
            idx = master_df.index[diffs.argmin()]
            for col in source_df.columns:
                if col in master_df.columns:
                    master_df.loc[idx, col] = source_df.loc[src_date, col]
            matched += 1
        else:
            unmatched += 1

    if fill == 'ffill':
        for col in new_cols:
            master_df[col] = master_df[col].ffill()

    print(f"  {label}: {matched} matched  {unmatched} unmatched  "
          f"{len(new_cols)} new columns")
    return master_df


print("=== Aligning to master meeting dates ===\n")
master = align_to_master(master, df_fred, fill=FILL_METHOD, label="FRED medians")
master = align_to_master(master, df_dots, fill=FILL_METHOD, label="Dot distribution")

# ══════════════════════════════════════════════════════════════════════════════
# STEP 6 — SUMMARY
# ══════════════════════════════════════════════════════════════════════════════

sep_cols = [c for c in master.columns if c.startswith('sep_')]
print(f"\n=== SEP columns in master: {len(sep_cols)} ===\n")

groups = {
    'Medians (central tendency)':   [c for c in sep_cols if 'median' in c],
    'IQR (committee dispersion)':   [c for c in sep_cols if '_iqr_' in c],
    'Bowley skewness (tail bias)':  [c for c in sep_cols if 'skew_bowley' in c],
    'Tail asymmetry':               [c for c in sep_cols if 'tail_asym' in c],
    'Range (min/max)':              [c for c in sep_cols if '_min_' in c
                                     or '_max_' in c or '_p25_' in c or '_p75_' in c],
    'Revisions':                    [c for c in sep_cols if 'revision' in c],
    'FRED medians':                 [c for c in sep_cols if 'med_cy' in c
                                     or 'med_1y' in c or 'med_2y' in c or 'med_lr' in c],
}

for grp, cols in groups.items():
    if cols:
        n_non = sum((master[c] != 0).sum() for c in cols) // len(cols) \
                if FILL_METHOD == 0 else \
                sum(master[c].notna().sum() for c in cols) // len(cols)
        print(f"  {grp} ({len(cols)} cols, avg ~{n_non} non-zero):")
        for c in cols:
            nz = (master[c] != 0).sum() if FILL_METHOD == 0 \
                 else master[c].notna().sum()
            v  = master[c].replace(0, np.nan)
            print(f"    {c:<48} n={nz:3d}  "
                  f"mean={v.mean():+.3f}  std={v.std():.3f}")
        print()

# ══════════════════════════════════════════════════════════════════════════════
# STEP 7 — SAVE
# ══════════════════════════════════════════════════════════════════════════════

master.index.name = 'meeting_date'
master.to_csv(OUTPUT_PATH)

print(f"=== Saved: {OUTPUT_PATH}  shape={master.shape} ===")
print(f"\nLoad with:")
print(f"  df_model = pd.read_csv('{OUTPUT_PATH}', "
      f"index_col='meeting_date', parse_dates=True)")
print()
print("Key variables for dispersion/skew analysis:")
print("  sep_ffr_iqr_{{cy,1y,2y}}         — committee disagreement (dispersion)")
print("  sep_ffr_skew_bowley_{{cy,1y,2y}} — hawkish/dovish tail (Bowley skew)")
print("  sep_ffr_tail_asym_{{cy,1y,2y}}   — upper vs lower extreme divergence")
print("  sep_ffr_revision_{{cy,1y}}        — meeting-over-meeting median shift")
print("  sep_ffr_revision_abs_{{cy,1y}}    — surprise magnitude")

=== Loading Master_Macro.csv ===
  Columns (first 10): ['meeting_date', 'event_type', 'cpi', 'cpi_yoy', 'core_cpi', 'core_cpi_yoy', 'pce', 'pce_yoy', 'core_pce', 'core_pce_yoy']
  363 rows  |  2011-01-08 -> 2025-12-31

=== Fetching SEP medians from FRED ===

  OK FEDTARMDLR: 57 obs  (2012-01-25 -> 2026-03-18)
  X FEDTARRLLO: HTTP 404
  X FEDTARRRHI: HTTP 404
  OK FEDTARCTL: 3 obs  (2026-01-01 -> 2028-01-01)
  OK FEDTARCTH: 3 obs  (2026-01-01 -> 2028-01-01)
  OK FEDTARCTM: 3 obs  (2026-01-01 -> 2028-01-01)

  Fetched 5 FRED series.

=== Scraping SEP dot distribution from federalreserve.gov ===

  Scraping 57 pages — this takes ~2 mins...

  [  1/57] OK 20120125  (36 stats)
  [  2/57] OK 20120425  (36 stats)
  [  3/57] OK 20120620  (36 stats)
  [  4/57] OK 20120913  (45 stats)
  [  5/57] X  20121212  HTTP 404
  [  6/57] OK 20130320  (36 stats)
  [  7/57] OK 20130619  (36 stats)
  [  8/57] OK 20130918  (45 stats)
  [  9/57] OK 20131218  (45 stats)
  [ 10/57] OK 20140319  (36 stats)
  [ 11

In [ ]:
"""
FOMC SEP Fetcher via FRED API
------------------------------
Pulls central tendency (midpoint, low, high), median, and range metrics
for GDP, unemployment, and PCE inflation from the FRED API.

Get a free API key at: https://fredaccount.stlouisfed.org/login/secure/

Usage:
    python fomc_sep_fred.py --api-key YOUR_KEY
    python fomc_sep_fred.py --api-key YOUR_KEY --output sep_data.csv
    python fomc_sep_fred.py --api-key YOUR_KEY --wide   # one row per meeting date

Dependencies:
    pip install requests pandas
"""

import argparse
import sys
import requests
import pandas as pd

FRED_BASE = "https://api.stlouisfed.org/fred/series/observations"

# ── Series catalogue ────────────────────────────────────────────────────────
# Each entry: (series_id, variable, horizon, metric)
#   horizon: "projection" = 2026-2028 annual, "longer_run" = long-run value
#   metric : median | ct_mid | ct_low | ct_high | range_mid | range_low | range_high

SERIES = [
    # ── Real GDP growth ──────────────────────────────────────────────────────
    ("GDPC1MD",   "gdp", "projection",  "median"),
    ("GDPC1CTM",  "gdp", "projection",  "ct_mid"),
    ("GDPC1CTL",  "gdp", "projection",  "ct_low"),
    ("GDPC1CTH",  "gdp", "projection",  "ct_high"),
    ("GDPC1RM",   "gdp", "projection",  "range_mid"),
    ("GDPC1RL",   "gdp", "projection",  "range_low"),
    ("GDPC1RH",   "gdp", "projection",  "range_high"),
    ("GDPC1MDLR",  "gdp", "longer_run", "median"),
    ("GDPC1CTMLR", "gdp", "longer_run", "ct_mid"),
    ("GDPC1CTLLR", "gdp", "longer_run", "ct_low"),
    ("GDPC1CTHLR", "gdp", "longer_run", "ct_high"),
    ("GDPC1RMLR",  "gdp", "longer_run", "range_mid"),
    ("GDPC1RLLR",  "gdp", "longer_run", "range_low"),
    ("GDPC1RHLR",  "gdp", "longer_run", "range_high"),

    # ── Unemployment rate ─────────────────────────────────────────────────────
    ("UNRATEMD",   "unemp", "projection",  "median"),
    ("UNRATECTM",  "unemp", "projection",  "ct_mid"),
    ("UNRATECTL",  "unemp", "projection",  "ct_low"),
    ("UNRATECTH",  "unemp", "projection",  "ct_high"),
    ("UNRATERL",   "unemp", "projection",  "range_low"),
    ("UNRATERH",   "unemp", "projection",  "range_high"),
    ("UNRATEMDLR",  "unemp", "longer_run", "median"),
    ("UNRATECTMLR", "unemp", "longer_run", "ct_mid"),
    ("UNRATERLLR",  "unemp", "longer_run", "range_low"),

    # ── PCE inflation ─────────────────────────────────────────────────────────
    ("PCECTPIMD",   "pce", "projection",  "median"),
    ("PCECTPICTM",  "pce", "projection",  "ct_mid"),
    ("PCECTPICTL",  "pce", "projection",  "ct_low"),
    ("PCECTPIRM",   "pce", "projection",  "range_mid"),
    ("PCECTPIRH",   "pce", "projection",  "range_high"),
    ("PCECTPIMDLR",  "pce", "longer_run", "median"),
    ("PCECTPICTMLR", "pce", "longer_run", "ct_mid"),
    ("PCECTPIRMLR",  "pce", "longer_run", "range_mid"),

    # ── Core PCE (ex food & energy) ───────────────────────────────────────────
    ("JCXFEMD",  "core_pce", "projection", "median"),
    ("JCXFECTM", "core_pce", "projection", "ct_mid"),
    ("JCXFECTH", "core_pce", "projection", "ct_high"),
    ("JCXFERM",  "core_pce", "projection", "range_mid"),
    ("JCXFERH",  "core_pce", "projection", "range_high"),
]


def fetch_series(series_id: str, api_key: str) -> pd.DataFrame:
    """Return a DataFrame with columns [date, value] for one FRED series."""
    params = {
        "series_id": series_id,
        "api_key": c5b5ec287d025fecd23b73051ee03c84,
        "file_type": "json",
    }
    resp = requests.get(FRED_BASE, params=params, timeout=15)
    resp.raise_for_status()
    data = resp.json()

    if "observations" not in data:
        raise ValueError(f"Unexpected response for {series_id}: {data}")

    df = pd.DataFrame(data["observations"])[["date", "value"]]
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df.rename(columns={"value": series_id}, inplace=True)
    return df


def build_long(api_key: str, verbose: bool = True) -> pd.DataFrame:
    """
    Fetch every series and return a tidy long-format DataFrame:
        date | variable | horizon | metric | series_id | value
    """
    frames = []
    for series_id, variable, horizon, metric in SERIES:
        if verbose:
            print(f"  {series_id:<18} {variable:<10} {horizon:<12} {metric}")
        try:
            df = fetch_series(series_id, api_key)
            df = df.rename(columns={series_id: "value"})
            df["series_id"] = series_id
            df["variable"] = variable
            df["horizon"] = horizon
            df["metric"] = metric
            frames.append(df)
        except Exception as e:
            print(f"  WARNING: could not fetch {series_id}: {e}", file=sys.stderr)

    if not frames:
        raise RuntimeError("No data fetched — check your API key.")

    long = pd.concat(frames, ignore_index=True)
    long = long[["date", "variable", "horizon", "metric", "series_id", "value"]]
    long = long.dropna(subset=["value"])
    long = long.sort_values(["date", "variable", "horizon", "metric"]).reset_index(drop=True)
    return long


def pivot_wide(long: pd.DataFrame) -> pd.DataFrame:
    """
    Pivot to wide format: one row per (date, horizon),
    columns like gdp_median, gdp_ct_mid, unemp_range_high, etc.
    """
    long = long.copy()
    long["col"] = long["variable"] + "_" + long["metric"]
    wide = long.pivot_table(
        index=["date", "horizon"],
        columns="col",
        values="value",
        aggfunc="first",
    ).reset_index()
    wide.columns.name = None

    # Re-order columns logically
    prefix_order = ["gdp", "unemp", "pce", "core_pce"]
    metric_order = ["median", "ct_low", "ct_mid", "ct_high",
                    "range_low", "range_mid", "range_high"]
    ordered_cols = ["date", "horizon"]
    for var in prefix_order:
        for metric in metric_order:
            col = f"{var}_{metric}"
            if col in wide.columns:
                ordered_cols.append(col)
    remaining = [c for c in wide.columns if c not in ordered_cols]
    wide = wide[ordered_cols + remaining]
    return wide


def main():
    parser = argparse.ArgumentParser(
        description="Fetch FOMC SEP data (GDP, unemployment, inflation) from FRED."
    )
    parser.add_argument("--api-key", required=True, help="FRED API key")
    parser.add_argument(
        "--output", default="fomc_sep.csv",
        help="Output filename (default: fomc_sep.csv)"
    )
    parser.add_argument(
        "--wide", action="store_true",
        help="Output wide format (one row per meeting date) instead of tidy long format"
    )
    parser.add_argument(
        "--quiet", action="store_true", help="Suppress per-series progress output"
    )
    args = parser.parse_args()

    print("Fetching SEP series from FRED...\n")
    long_df = build_long(api_key=args.api_key, verbose=not args.quiet)

    if args.wide:
        output_df = pivot_wide(long_df)
    else:
        output_df = long_df

    output_df.to_csv(args.output, index=False)
    print(f"\nSaved {len(output_df)} rows × {len(output_df.columns)} columns → {args.output}")
    print("\nSample (most recent observations):")
    print(output_df.tail(10).to_string(index=False))


if __name__ == "__main__":
    main()

usage: colab_kernel_launcher.py [-h] --api-key API_KEY [--output OUTPUT]
                                [--wide] [--quiet]
colab_kernel_launcher.py: error: the following arguments are required: --api-key
ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/lib/python3.12/argparse.py", line 1943, in _parse_known_args2
    namespace, args = self._parse_known_args(args, namespace, intermixed)
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/argparse.py", line 2230, in _parse_known_args
    raise ArgumentError(None, _('the following arguments are required: %s') %
argparse.ArgumentError: the following arguments are required: --api-key

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_1347/1221015694.py", line 190, in <cell line: 0>
    main()
  File "/tmp/ipykernel_1347/1221015694.py", line 173, in main
    args = parser.parse_args()
           ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/argparse.py", line

TypeError: object of type 'NoneType' has no len()

In [ ]:
API_KEY = "c5b5ec287d025fecd23b73051ee03c84"
OUTPUT  = "fomc_sep.csv"
WIDE    = False   # set True for the wide table layout
"""
FOMC SEP Fetcher via FRED API
------------------------------
Pulls central tendency (midpoint, low, high), median, and range metrics
for GDP, unemployment, and PCE inflation from the FRED API.

Get a free API key at: https://fredaccount.stlouisfed.org/login/secure/

── Config ──────────────────────────────────────────────────────────────────
Edit the three variables below, then run the cell.
"""

# ▼ EDIT THESE ▼
API_KEY = "c5b5ec287d025fecd23b73051ee03c84"   # https://fredaccount.stlouisfed.org/login/secure/
OUTPUT  = "fomc_sep.csv"        # output filename
WIDE    = False                 # True = one row per meeting date; False = tidy long format
# ▲ EDIT THESE ▲

import sys
import requests
import pandas as pd

FRED_BASE = "https://api.stlouisfed.org/fred/series/observations"

# ── Series catalogue ────────────────────────────────────────────────────────
# Each entry: (series_id, variable, horizon, metric)
#   horizon: "projection" = 2026-2028 annual, "longer_run" = long-run value
#   metric : median | ct_mid | ct_low | ct_high | range_mid | range_low | range_high

SERIES = [
    # ── Real GDP growth ──────────────────────────────────────────────────────
    ("GDPC1MD",   "gdp", "projection",  "median"),
    ("GDPC1CTM",  "gdp", "projection",  "ct_mid"),
    ("GDPC1CTL",  "gdp", "projection",  "ct_low"),
    ("GDPC1CTH",  "gdp", "projection",  "ct_high"),
    ("GDPC1RM",   "gdp", "projection",  "range_mid"),
    ("GDPC1RL",   "gdp", "projection",  "range_low"),
    ("GDPC1RH",   "gdp", "projection",  "range_high"),
    ("GDPC1MDLR",  "gdp", "longer_run", "median"),
    ("GDPC1CTMLR", "gdp", "longer_run", "ct_mid"),
    ("GDPC1CTLLR", "gdp", "longer_run", "ct_low"),
    ("GDPC1CTHLR", "gdp", "longer_run", "ct_high"),
    ("GDPC1RMLR",  "gdp", "longer_run", "range_mid"),
    ("GDPC1RLLR",  "gdp", "longer_run", "range_low"),
    ("GDPC1RHLR",  "gdp", "longer_run", "range_high"),

    # ── Unemployment rate ─────────────────────────────────────────────────────
    ("UNRATEMD",   "unemp", "projection",  "median"),
    ("UNRATECTM",  "unemp", "projection",  "ct_mid"),
    ("UNRATECTL",  "unemp", "projection",  "ct_low"),
    ("UNRATECTH",  "unemp", "projection",  "ct_high"),
    ("UNRATERL",   "unemp", "projection",  "range_low"),
    ("UNRATERH",   "unemp", "projection",  "range_high"),
    ("UNRATEMDLR",  "unemp", "longer_run", "median"),
    ("UNRATECTMLR", "unemp", "longer_run", "ct_mid"),
    ("UNRATERLLR",  "unemp", "longer_run", "range_low"),

    # ── PCE inflation ─────────────────────────────────────────────────────────
    ("PCECTPIMD",   "pce", "projection",  "median"),
    ("PCECTPICTM",  "pce", "projection",  "ct_mid"),
    ("PCECTPICTL",  "pce", "projection",  "ct_low"),
    ("PCECTPIRM",   "pce", "projection",  "range_mid"),
    ("PCECTPIRH",   "pce", "projection",  "range_high"),
    ("PCECTPIMDLR",  "pce", "longer_run", "median"),
    ("PCECTPICTMLR", "pce", "longer_run", "ct_mid"),
    ("PCECTPIRMLR",  "pce", "longer_run", "range_mid"),

    # ── Core PCE (ex food & energy) ───────────────────────────────────────────
    ("JCXFEMD",  "core_pce", "projection", "median"),
    ("JCXFECTM", "core_pce", "projection", "ct_mid"),
    ("JCXFECTH", "core_pce", "projection", "ct_high"),
    ("JCXFERM",  "core_pce", "projection", "range_mid"),
    ("JCXFERH",  "core_pce", "projection", "range_high"),
]


def fetch_series(series_id: str, api_key: str) -> pd.DataFrame:
    """Return a DataFrame with columns [date, value] for one FRED series."""
    params = {
        "series_id": series_id,
        "api_key": api_key,
        "file_type": "json",
    }
    resp = requests.get(FRED_BASE, params=params, timeout=15)
    resp.raise_for_status()
    data = resp.json()

    if "observations" not in data:
        raise ValueError(f"Unexpected response for {series_id}: {data}")

    df = pd.DataFrame(data["observations"])[["date", "value"]]
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df.rename(columns={"value": series_id}, inplace=True)
    return df


def build_long(api_key: str, verbose: bool = True) -> pd.DataFrame:
    """
    Fetch every series and return a tidy long-format DataFrame:
        date | variable | horizon | metric | series_id | value
    """
    frames = []
    for series_id, variable, horizon, metric in SERIES:
        if verbose:
            print(f"  {series_id:<18} {variable:<10} {horizon:<12} {metric}")
        try:
            df = fetch_series(series_id, api_key)
            df = df.rename(columns={series_id: "value"})
            df["series_id"] = series_id
            df["variable"] = variable
            df["horizon"] = horizon
            df["metric"] = metric
            frames.append(df)
        except Exception as e:
            print(f"  WARNING: could not fetch {series_id}: {e}", file=sys.stderr)

    if not frames:
        raise RuntimeError("No data fetched — check your API key.")

    long = pd.concat(frames, ignore_index=True)
    long = long[["date", "variable", "horizon", "metric", "series_id", "value"]]
    long = long.dropna(subset=["value"])
    long = long.sort_values(["date", "variable", "horizon", "metric"]).reset_index(drop=True)
    return long


def pivot_wide(long: pd.DataFrame) -> pd.DataFrame:
    """
    Pivot to wide format: one row per (date, horizon),
    columns like gdp_median, gdp_ct_mid, unemp_range_high, etc.
    """
    long = long.copy()
    long["col"] = long["variable"] + "_" + long["metric"]
    wide = long.pivot_table(
        index=["date", "horizon"],
        columns="col",
        values="value",
        aggfunc="first",
    ).reset_index()
    wide.columns.name = None

    # Re-order columns logically
    prefix_order = ["gdp", "unemp", "pce", "core_pce"]
    metric_order = ["median", "ct_low", "ct_mid", "ct_high",
                    "range_low", "range_mid", "range_high"]
    ordered_cols = ["date", "horizon"]
    for var in prefix_order:
        for metric in metric_order:
            col = f"{var}_{metric}"
            if col in wide.columns:
                ordered_cols.append(col)
    remaining = [c for c in wide.columns if c not in ordered_cols]
    wide = wide[ordered_cols + remaining]
    return wide


if API_KEY == "YOUR_FRED_API_KEY":
    raise ValueError("Please set API_KEY at the top of the script.")

print("Fetching SEP series from FRED...\n")
long_df = build_long(api_key=API_KEY, verbose=True)

output_df = pivot_wide(long_df) if WIDE else long_df

output_df.to_csv(OUTPUT, index=False)
print(f"\nSaved {len(output_df)} rows × {len(output_df.columns)} columns → {OUTPUT}")
print("\nSample (most recent observations):")
print(output_df.tail(10).to_string(index=False))

Fetching SEP series from FRED...

  GDPC1MD            gdp        projection   median
  GDPC1CTM           gdp        projection   ct_mid
  GDPC1CTL           gdp        projection   ct_low
  GDPC1CTH           gdp        projection   ct_high
  GDPC1RM            gdp        projection   range_mid
  GDPC1RL            gdp        projection   range_low
  GDPC1RH            gdp        projection   range_high
  GDPC1MDLR          gdp        longer_run   median
  GDPC1CTMLR         gdp        longer_run   ct_mid
  GDPC1CTLLR         gdp        longer_run   ct_low
  GDPC1CTHLR         gdp        longer_run   ct_high
  GDPC1RMLR          gdp        longer_run   range_mid
  GDPC1RLLR          gdp        longer_run   range_low
  GDPC1RHLR          gdp        longer_run   range_high
  UNRATEMD           unemp      projection   median
  UNRATECTM          unemp      projection   ct_mid
  UNRATECTL          unemp      projection   ct_low
  UNRATECTH          unemp      projection   ct_high
  UNRAT

In [ ]:
output_df

,date,variable,horizon,metric,series_id,value
0,2009-02-18,gdp,longer_run,ct_high,GDPC1CTHLR,2.7
1,2009-02-18,gdp,longer_run,ct_low,GDPC1CTLLR,2.5
2,2009-02-18,gdp,longer_run,ct_mid,GDPC1CTMLR,2.6
3,2009-02-18,gdp,longer_run,range_high,GDPC1RHLR,3.0
4,2009-02-18,gdp,longer_run,range_low,GDPC1RLLR,2.4
...,...,...,...,...,...,...
883,2028-01-01,unemp,projection,ct_low,UNRATECTL,4.0
884,2028-01-01,unemp,projection,ct_mid,UNRATECTM,4.2
885,2028-01-01,unemp,projection,median,UNRATEMD,4.2
886,2028-01-01,unemp,projection,range_high,UNRATERH,4.5


In [ ]:
"""
Merge SPF Dispersion Data into Master_Macro.csv
================================================
For each date in Master_Macro.csv, this script:
  1. Converts the date to the corresponding SPF survey quarter
  2. Looks up the D1/D2/D3 dispersion values for that quarter
  3. Adds the relevant columns to the CSV

Variables extracted:
  - RGDP  : Real GDP               -> D1 (levels) + D2 (Q/Q growth)
  - NGDP  : Nominal GDP            -> D1 (levels) + D2 (Q/Q growth)
  - UNEMP : Unemployment rate      -> D1 (levels)
  - CPI   : CPI inflation          -> D1 only (not in D2/D3 by design)
  - PCE   : PCE inflation          -> D1 only (not in D2/D3 by design)
  - RCONSUM: Real consumption      -> D1 (levels) + D2 (Q/Q growth)

NOTE: Adjust FILE PATHS below to match where you saved the Excel files.
"""

import pandas as pd
import numpy as np
from pathlib import Path


# =============================================================================
# FILE PATHS — adjust these to match your local setup
# =============================================================================
MASTER_CSV      = r"/content/drive/MyDrive/HEC Thesis/Data/Master_Macro.csv"
D1_FILE         = r"/content/drive/MyDrive/HEC Thesis/Data/Dispersion_1.xlsx"      # Dispersion for levels (includes CPI, PCE)
D2_FILE         = r"/content/drive/MyDrive/HEC Thesis/Data/Dispersion_2.xlsx"    # Dispersion for Q/Q growth (excludes CPI, PCE)
D3_FILE         = r"/content/drive/MyDrive/HEC Thesis/Data/Dispersion_3.xlsx"    # Dispersion for log differences
OUTPUT_CSV      = r"/content/drive/MyDrive/HEC Thesis/Data/Master_Macro.csv"

# SPF variable codes to extract (sheet names inside the Excel files)
# D1 variables (levels — includes inflation measures)
D1_VARS = ["RGDP", "NGDP", "UNEMP", "CPI", "PCE", "RCONSUM"]

# D2 variables (Q/Q growth — CPI and PCE are excluded by the Fed)
D2_VARS = ["RGDP", "NGDP", "UNEMP", "RCONSUM"]

# D3 variables (log differences — same exclusions as D2)
D3_VARS = ["RGDP", "NGDP", "UNEMP", "RCONSUM"]

# Forecast horizons to pull: T = current quarter, T+1 = next quarter, etc.
HORIZONS = ["T", "T+1", "T+2", "T+3", "T+4"]


# =============================================================================
# HELPER: parse SPF survey date string (e.g. "1968Q4") into (year, quarter)
# =============================================================================
def parse_spf_date(s: str) -> tuple[int, int]:
    s = str(s).strip()
    year = int(s[:4])
    quarter = int(s[5])
    return year, quarter


# =============================================================================
# HELPER: convert a calendar date to its SPF survey quarter
# The SPF survey is sent AFTER the BEA advance release for that quarter,
# so a date in quarter Q belongs to SPF survey Q.
# =============================================================================
def date_to_spf_quarter(date: pd.Timestamp) -> str:
    quarter = (date.month - 1) // 3 + 1
    return f"{date.year}Q{quarter}"


# =============================================================================
# LOAD one sheet from a dispersion Excel file and reshape into a lookup table
# Returns a DataFrame indexed by survey_quarter (e.g. "1968Q4") with columns:
#   {VAR}_P25_D{n}(T), {VAR}_P75_D{n}(T), {VAR}_D{n}(T), ... for each horizon
# =============================================================================
def load_dispersion_sheet(
    filepath: str,
    variable: str,
    measure: str,   # "D1", "D2", or "D3"
) -> pd.DataFrame | None:
    path = Path(filepath)
    if not path.exists():
        print(f"  [WARNING] File not found: {filepath}")
        return None

    try:
        xl = pd.ExcelFile(filepath)
    except Exception as e:
        print(f"  [ERROR] Could not open {filepath}: {e}")
        return None

    if variable not in xl.sheet_names:
        print(f"  [WARNING] Sheet '{variable}' not found in {filepath}. "
              f"Available: {xl.sheet_names}")
        return None

    # Read sheet, skip metadata rows (rows 0-8 typically), use row 9 as header
    # The Fed files have ~9 header rows before the data starts
    df_raw = xl.parse(variable, header=None)

    # Find the row where "Survey_Date(T)" appears — that's the column header row
    header_row = None
    for i, row in df_raw.iterrows():
        if any("Survey_Date" in str(v) for v in row.values):
            header_row = i
            break

    if header_row is None:
        print(f"  [WARNING] Could not find header row in {variable} sheet of {filepath}")
        return None

    df = xl.parse(variable, header=header_row)
    df.columns = [str(c).strip() for c in df.columns]

    # Drop rows where Survey_Date is NaN
    date_col = [c for c in df.columns if "Survey_Date" in c]
    if not date_col:
        print(f"  [WARNING] No Survey_Date column found in {variable}")
        return None
    date_col = date_col[0]
    df = df.dropna(subset=[date_col])
    df[date_col] = df[date_col].astype(str).str.strip()

    # Use Survey_Date as the index
    df = df.set_index(date_col)

    print(f"  Loaded {variable} from {measure} ({filepath}): "
          f"{len(df)} quarters, {len(df.columns)} columns")
    return df


# =============================================================================
# BUILD a single lookup DataFrame for all variables from one dispersion file
# =============================================================================
def build_lookup(filepath: str, variables: list[str], measure: str) -> pd.DataFrame:
    all_sheets = []
    for var in variables:
        df = load_dispersion_sheet(filepath, var, measure)
        if df is not None:
            # Prefix columns with variable name + measure if not already
            # (the Fed files already name columns like NGDP_P25_D1(T) etc.)
            all_sheets.append(df)

    if not all_sheets:
        return pd.DataFrame()

    # Join all variable sheets on the survey quarter index
    combined = all_sheets[0]
    for sheet in all_sheets[1:]:
        combined = combined.join(sheet, how="outer", rsuffix="_dup")
        # Drop any duplicate columns that sneak in
        combined = combined[[c for c in combined.columns if not c.endswith("_dup")]]

    return combined


# =============================================================================
# MAIN
# =============================================================================
def main():
    # -------------------------------------------------------------------------
    # 1. Load Master_Macro.csv
    # -------------------------------------------------------------------------
    print(f"\nLoading {MASTER_CSV} ...")
    master = pd.read_csv(MASTER_CSV, parse_dates=["date"])
    print(f"  {len(master)} rows, {len(master.columns)} columns")

    # Add a column for the SPF survey quarter corresponding to each date
    master["spf_quarter"] = master["date"].apply(date_to_spf_quarter)
    print(f"  Quarter range: {master['spf_quarter'].min()} to {master['spf_quarter'].max()}")

    # -------------------------------------------------------------------------
    # 2. Load dispersion data
    # -------------------------------------------------------------------------
    print("\nLoading D1 (levels, includes CPI/PCE) ...")
    d1_lookup = build_lookup(D1_FILE, D1_VARS, "D1")

    print("\nLoading D2 (Q/Q growth, excludes CPI/PCE) ...")
    d2_lookup = build_lookup(D2_FILE, D2_VARS, "D2")

    print("\nLoading D3 (log differences) ...")
    d3_lookup = build_lookup(D3_FILE, D3_VARS, "D3")

    # -------------------------------------------------------------------------
    # 3. Merge into master on the survey quarter
    # -------------------------------------------------------------------------
    print("\nMerging SPF data into master ...")

    def merge_lookup(master_df, lookup_df, label):
        if lookup_df.empty:
            print(f"  [SKIP] {label} lookup is empty.")
            return master_df
        merged = master_df.join(
            lookup_df,
            on="spf_quarter",
            how="left",
            rsuffix=f"_{label}_dup"
        )
        # Drop duplicate columns
        merged = merged[[c for c in merged.columns if not c.endswith("_dup")]]
        matched = merged[lookup_df.columns[0]].notna().sum()
        print(f"  {label}: {matched}/{len(master_df)} rows matched")
        return merged

    master = merge_lookup(master, d1_lookup, "D1")
    master = merge_lookup(master, d2_lookup, "D2")
    master = merge_lookup(master, d3_lookup, "D3")

    # -------------------------------------------------------------------------
    # 4. Drop the helper column and save
    # -------------------------------------------------------------------------
    # Keep spf_quarter as it's useful context
    print(f"\nSaving to {OUTPUT_CSV} ...")
    master.to_csv(OUTPUT_CSV, index=False)
    print(f"  Done. Output: {len(master)} rows x {len(master.columns)} columns")

    # -------------------------------------------------------------------------
    # 5. Quick summary of added columns
    # -------------------------------------------------------------------------
    original_cols = set(pd.read_csv(MASTER_CSV, nrows=0).columns) | {"spf_quarter"}
    new_cols = [c for c in master.columns if c not in original_cols]
    print(f"\nNew SPF columns added ({len(new_cols)}):")
    for col in new_cols:
        print(f"  {col}")

    return master


if __name__ == "__main__":
    result = main()
    print("\nPreview of merged data (first 3 rows, SPF columns only):")
    original_cols = set(pd.read_csv(MASTER_CSV, nrows=0).columns) | {"spf_quarter"}
    spf_cols = ["date", "event_type", "spf_quarter"] + \
               [c for c in result.columns if c not in original_cols]
    print(result[spf_cols].head(3).to_string())


Loading /content/drive/MyDrive/HEC Thesis/Data/Master_Macro.csv ...
  363 rows, 56 columns
  Quarter range: 2011Q1 to 2025Q4

Loading D1 (levels, includes CPI/PCE) ...
  [WARNING] Sheet 'RGDP' not found in /content/drive/MyDrive/HEC Thesis/Data/Dispersion_1.xlsx. Available: ['NGDP', 'CPROF', 'UNEMP', 'EMP', 'HOUSING', 'TBILL', 'BOND', 'BAABOND', 'TBOND', 'CPI', 'CORECPI', 'PCE', 'COREPCE', 'CPI5YR', 'PCE5YR', 'CPI10', 'PCE10', 'RGDP10', 'PROD10', 'STOCK10', 'BOND10', 'BILL10', 'UBAR', 'SPR_Tbond_Tbill', 'SPR_BAA_AAA', 'SPR_BAA_TBOND', 'SPR_AAA_TBOND', 'CPIF5', 'PCEF5', 'RR1_TBILL_PGDP', 'RR2_TBILL_PGDP', 'RR3_TBILL_PGDP', 'RR1_TBILL_CPI', 'RR2_TBILL_CPI', 'RR3_TBILL_CPI', 'RR1_TBILL_CCPI', 'RR2_TBILL_CCPI', 'RR3_TBILL_CCPI', 'RR1_TBILL_PCE', 'RR2_TBILL_PCE', 'RR3_TBILL_PCE', 'RR1_TBILL_CPCE', 'RR2_TBILL_CPCE', 'RR3_TBILL_CPCE']
  Loaded NGDP from D1 (/content/drive/MyDrive/HEC Thesis/Data/Dispersion_1.xlsx): 230 quarters, 15 columns
  Loaded UNEMP from D1 (/content/drive/MyDrive/HEC T

In [7]:
!pip install fredapi pandas
!pip install pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 67.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 184.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 284.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 21.1 MB/s eta 0:00:00


In [4]:
"""
FOMC SEP Dispersion Metrics via FRED API
-----------------------------------------
Get a free API key at: https://fred.stlouisfed.org/docs/api/api_key.html
Install deps: pip install fredapi pandas
"""

import pandas as pd
from fredapi import Fred

API_KEY = "c5b5ec287d025fecd23b73051ee03c84"  # <-- replace this
fred = Fred(api_key=API_KEY)

# ── Series definitions ────────────────────────────────────────────────────────

SERIES = {
    # Medians (not dispersion, but useful as anchors)
    "median_current":     "FEDTARMD0",
    "median_1y":          "FEDTARMD1",
    "median_2y":          "FEDTARMD2",
    "median_3y":          "FEDTARMD3",
    "median_longer_run":  "FEDTARMD",

    # Central tendency bounds (trimmed: 3 highest + 3 lowest removed)
    # HIGH - LOW = dispersion among the "core" participants
    "ct_high":            "FEDTARCTH",
    "ct_low":             "FEDTARCTL",

    # Full range (all participants, including outliers)
    # HIGH - LOW = total dispersion across the committee
    "range_high":         "FEDTARRH",
    "range_low":          "FEDTARRL",
}

# ── Fetch ─────────────────────────────────────────────────────────────────────

def fetch_all(series_map: dict) -> pd.DataFrame:
    frames = {}
    for name, fred_id in series_map.items():
        try:
            s = fred.get_series(fred_id)
            s.name = name
            frames[name] = s
            print(f"  ✓ {fred_id:15s} ({name})")
        except Exception as e:
            print(f"  ✗ {fred_id:15s} — {e}")

    df = pd.DataFrame(frames)
    df.index = pd.to_datetime(df.index)
    df.index.name = "date"
    return df.dropna(how="all")

# ── Compute dispersion metrics ────────────────────────────────────────────────

def add_dispersion(df: pd.DataFrame) -> pd.DataFrame:
    """
    Two flavours of dispersion, both available from FRED:

    1. Central tendency spread  — committee view without the 3 outliers on each end
    2. Full range spread        — total disagreement including outlier participants

    These are the closest thing to dispersion available without the raw dot data.
    """
    df = df.copy()

    if {"ct_high", "ct_low"}.issubset(df.columns):
        df["ct_spread"] = df["ct_high"] - df["ct_low"]

    if {"range_high", "range_low"}.issubset(df.columns):
        df["full_spread"] = df["range_high"] - df["range_low"]

    # Disagreement index: how much wider is the full range vs the core?
    # A large gap means outlier participants are pulling far from consensus.
    if {"ct_spread", "full_spread"}.issubset(df.columns):
        df["outlier_pull"] = df["full_spread"] - df["ct_spread"]

    return df

# ── Run ───────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    print("Fetching SEP series from FRED...")
    df = fetch_all(SERIES)
    df = add_dispersion(df)

    print(f"\nData shape: {df.shape}")
    print(f"Date range: {df.index.min().date()} → {df.index.max().date()}")
    print(f"\nMeeting count: {len(df)}")  # ~4 per year since 2012

    print("\nLatest observation:")
    print(df.tail(1).T.to_string())

    print("\nDispersion metrics (last 5 meetings):")
    disp_cols = ["median_current", "ct_spread", "full_spread", "outlier_pull"]
    available = [c for c in disp_cols if c in df.columns]
    print(df[available].tail(5).to_string())

    # Save to CSV
    out_path = "sep_dispersion.csv"
    df.to_csv(out_path)
    print(f"\nSaved to {out_path}")

Fetching SEP series from FRED...
  ✗ FEDTARMD0       — Bad Request.  The series does not exist.
  ✗ FEDTARMD1       — Bad Request.  The series does not exist.
  ✗ FEDTARMD2       — Bad Request.  The series does not exist.
  ✗ FEDTARMD3       — Bad Request.  The series does not exist.
  ✓ FEDTARMD        (median_longer_run)
  ✓ FEDTARCTH       (ct_high)
  ✓ FEDTARCTL       (ct_low)
  ✓ FEDTARRH        (range_high)
  ✓ FEDTARRL        (range_low)

Data shape: (3, 8)
Date range: 2026-01-01 → 2028-01-01

Meeting count: 3

Latest observation:
date               2028-01-01
median_longer_run         3.1
ct_high                   3.6
ct_low                    2.9
range_high                3.9
range_low                 2.6
ct_spread                 0.7
full_spread               1.3
outlier_pull              0.6

Dispersion metrics (last 5 meetings):
            ct_spread  full_spread  outlier_pull
date                                            
2026-01-01        0.5          1.0           0.5


In [5]:
df[available].head()

,ct_spread,full_spread,outlier_pull
date,,,
2026-01-01,0.5,1.0,0.5
2027-01-01,0.7,1.5,0.8
2028-01-01,0.7,1.3,0.6


In [10]:
"""
FOMC SEP Historical Scraper — Fixed Parser
-------------------------------------------
Era breakdown:
  pre-2012  : No fed funds rate projections in SEP
  2012–2017 : Fed funds only in dot plot figure (not Table 1) — requires dot scraping
  2018+     : Fed funds in Table 1 memo row, ranges use en-dash e.g. "2.1–2.4"

This script reliably handles 2018+ (Table 1 parsing).
For 2012-2017, it falls back to extracting range/CT from the dot plot page text.

Install: pip install pdfplumber requests pandas
"""

import re
import time
import requests
import pdfplumber
import pandas as pd
from io import BytesIO

# ── Meeting dates (2018 onwards for reliable Table 1 parsing) ─────────────────

SEP_MEETINGS_2018_PLUS = [
    "20180321", "20180613", "20180926", "20181219",
    "20190320", "20190619", "20190918", "20191211",
    "20200129", "20200611", "20200916", "20201216",
    "20210317", "20210616", "20210922", "20211215",
    "20220316", "20220615", "20220921", "20221214",
    "20230322", "20230614", "20230920", "20231213",
    "20240320", "20240612", "20240918", "20241218",
    "20250319", "20250618", "20250917", "20251210",
]

# 2012–2017: fed funds only in dot plot, harder to parse
SEP_MEETINGS_2012_2017 = [
    "20120125", "20120425", "20121212",
    "20130320", "20130619", "20130918", "20131218",
    "20140319", "20140618", "20140917", "20141217",
    "20150318", "20150617", "20150917", "20151216",
    "20160316", "20160615", "20160921", "20161214",
    "20170315", "20170614", "20170920", "20171213",
]

BASE_URL = "https://www.federalreserve.gov/monetarypolicy/files/fomcprojtabl{}.pdf"

# ── Fetch PDF ─────────────────────────────────────────────────────────────────

def fetch_pdf(date_str: str) -> bytes | None:
    url = BASE_URL.format(date_str)
    try:
        r = requests.get(url, timeout=20, headers={"User-Agent": "Mozilla/5.0"})
        if r.status_code == 200:
            return r.content
        return None
    except Exception as e:
        print(f"  Error fetching {date_str}: {e}")
        return None

def get_page_text(pdf_bytes: bytes, page_idx: int = 0) -> str:
    with pdfplumber.open(BytesIO(pdf_bytes)) as pdf:
        if page_idx >= len(pdf.pages):
            return ""
        return pdf.pages[page_idx].extract_text() or ""

# ── Number parsing helpers ────────────────────────────────────────────────────

def parse_range(s: str) -> tuple[float, float] | None:
    """Parse 'X.X–Y.Y' or 'X.X-Y.Y' or 'X.X to Y.Y' into (low, high)."""
    s = s.strip()
    # en-dash or hyphen range
    m = re.match(r'^(\d+\.?\d*)\s*[–\-]\s*(\d+\.?\d*)$', s)
    if m:
        return float(m.group(1)), float(m.group(2))
    # "X to Y" format
    m = re.match(r'^(\d+\.?\d*)\s+to\s+(\d+\.?\d*)$', s)
    if m:
        return float(m.group(1)), float(m.group(2))
    return None

def extract_numbers(text: str) -> list[float]:
    """Extract all standalone decimal numbers from a string."""
    return [float(x) for x in re.findall(r'\b\d+\.\d+\b', text)]

# ── Parser: 2018+ (Table 1, Memo row) ────────────────────────────────────────
#
# Table structure (page 1 or 2):
#   Median | Central Tendency | Range
#   [yr0] [yr1] [yr2] [LR] | [lo–hi] x4 | [lo–hi] x4
#
# The "Federalfundsrate" line contains all 12 values on one line.

def parse_2018_plus(date_str: str, pdf_bytes: bytes) -> list[dict]:
    text = get_page_text(pdf_bytes, 0)
    if not text:
        text = get_page_text(pdf_bytes, 1)

    # Normalize: remove line breaks within the funds rate line
    # Find the funds rate memo block
    # Pattern: "Federalfundsrate" or "Federal funds rate" followed by numbers
    text_clean = re.sub(r'\s+', ' ', text)

    # Find the Federal funds rate line
    m = re.search(
        r'[Ff]ederal\s*funds\s*rate\s+'  # label
        r'([\d\s.–\-]+?)'                # all numbers on this row
        r'(?:[Dd]ecember|[Mm]arch|[Jj]une|[Ss]eptember|[Nn]ovember|[Jj]anuary|Note:|$)',
        text_clean
    )
    if not m:
        return []

    raw = m.group(1).strip()

    # Split on whitespace — tokens are either single numbers or "X.X–Y.Y"
    tokens = raw.split()

    # Separate single values (medians) from ranges
    medians = []
    ranges = []
    for tok in tokens:
        r = parse_range(tok)
        if r:
            ranges.append(r)
        else:
            try:
                medians.append(float(tok))
            except ValueError:
                pass

    # Expected: 4 medians, 4 CT ranges, 4 full ranges
    # Horizons: current_year, 1y_ahead, 2y_ahead, longer_run
    horizon_labels = ["current_year", "1y_ahead", "2y_ahead", "longer_run"]
    n_horizons = min(4, len(ranges) // 2)

    ct_ranges = ranges[:n_horizons]
    full_ranges = ranges[n_horizons:n_horizons*2]

    rows = []
    for i in range(n_horizons):
        row = {
            "meeting_date": pd.Timestamp(f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:]}"),
            "horizon": horizon_labels[i] if i < len(horizon_labels) else f"h{i}",
        }
        if i < len(medians):
            row["median"] = medians[i]
        if i < len(ct_ranges):
            row["ct_low"], row["ct_high"] = ct_ranges[i]
            row["ct_spread"] = round(ct_ranges[i][1] - ct_ranges[i][0], 3)
            row["ct_midpoint"] = round(sum(ct_ranges[i]) / 2, 3)
        if i < len(full_ranges):
            row["range_low"], row["range_high"] = full_ranges[i]
            row["full_spread"] = round(full_ranges[i][1] - full_ranges[i][0], 3)
        if "ct_spread" in row and "full_spread" in row:
            row["outlier_pull"] = round(row["full_spread"] - row["ct_spread"], 3)
        rows.append(row)

    return rows

# ── Parser: 2012–2017 (range only from Table 1 — GDP/unemployment/inflation) ──
#
# Fed funds rate is only in the dot plot for this era.
# We can still get CT and range from the projection table for OTHER variables,
# but for fed funds we need to read the dot plot page text.
# The dot plot page has text like "Target federal funds rate at year end" with
# axis labels that give approximate range info.

def parse_2012_2017(date_str: str, pdf_bytes: bytes) -> list[dict]:
    """
    For 2012-2017, extract fed funds range from the dot plot page.
    The page text contains the y-axis scale values which bracket all dots.
    This gives us an approximate full range but NOT central tendency.
    """
    with pdfplumber.open(BytesIO(pdf_bytes)) as pdf:
        all_text = "\n".join(p.extract_text() or "" for p in pdf.pages)

    # Find dot plot section
    m = re.search(
        r'(?:Appropriate pace|Target [Ff]ederal funds rate at year.?end)(.*?)(?:NOTE|Note|Explanation)',
        all_text, re.DOTALL
    )
    if not m:
        return []

    block = m.group(1)
    # Extract all numbers — the y-axis labels give us the scale
    nums = sorted(set(extract_numbers(block)))
    if len(nums) < 2:
        return []

    # The numbers visible on the dot plot page are y-axis gridlines
    # Approximate range: min to max of axis labels
    row = {
        "meeting_date": pd.Timestamp(f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:]}"),
        "horizon": "dot_plot_axis_range",  # approximate only
        "range_low": nums[0],
        "range_high": nums[-1],
        "full_spread": round(nums[-1] - nums[0], 3),
        "note": "approx from dot plot axis labels — no CT available for this era",
    }
    return [row]

# ── Main scrape loop ──────────────────────────────────────────────────────────

def scrape_all(meetings: list[str], parser_fn, label: str, delay: float = 1.0) -> pd.DataFrame:
    all_rows = []
    for i, date_str in enumerate(meetings):
        print(f"[{i+1:02d}/{len(meetings)}] {date_str} ", end="", flush=True)
        pdf_bytes = fetch_pdf(date_str)
        if pdf_bytes is None:
            print("✗ 404")
            continue
        rows = parser_fn(date_str, pdf_bytes)
        if rows:
            all_rows.extend(rows)
            print(f"✓  {len(rows)} horizons")
        else:
            print("⚠  parse failed")
        time.sleep(delay)

    return pd.DataFrame(all_rows) if all_rows else pd.DataFrame()

# ── Run ───────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    print("=== Phase 1: 2018+ (Table 1 — reliable) ===")
    df_2018 = scrape_all(SEP_MEETINGS_2018_PLUS, parse_2018_plus, "2018+")

    print("\n=== Phase 2: 2012–2017 (dot plot approximate) ===")
    df_2012 = scrape_all(SEP_MEETINGS_2012_2017, parse_2012_2017, "2012-2017")

    if not df_2018.empty:
        df_2018.to_csv("sep_2018_plus.csv", index=False)
        print(f"\n✓ Saved sep_2018_plus.csv  ({len(df_2018)} rows, "
              f"{df_2018['meeting_date'].nunique()} meetings)")

        print("\nSample — current year horizon:")
        cy = df_2018[df_2018["horizon"] == "current_year"]
        print(cy[["meeting_date", "median", "ct_spread", "full_spread", "outlier_pull"]].to_string(index=False))

    if not df_2012.empty:
        df_2012.to_csv("sep_2012_2017.csv", index=False)
        print(f"\n✓ Saved sep_2012_2017.csv  ({len(df_2012)} rows — approximate)")

=== Phase 1: 2018+ (Table 1 — reliable) ===
[01/32] 20180321 ✓  4 horizons
[02/32] 20180613 ✓  4 horizons
[03/32] 20180926 ✓  4 horizons
[04/32] 20181219 ✓  4 horizons
[05/32] 20190320 ✓  4 horizons
[06/32] 20190619 ✓  4 horizons
[07/32] 20190918 ✓  4 horizons
[08/32] 20191211 ✓  4 horizons
[09/32] 20200129 ✗ 404
[10/32] 20200611 ✗ 404
[11/32] 20200916 ✓  2 horizons
[12/32] 20201216 ⚠  parse failed
[13/32] 20210317 ⚠  parse failed
[14/32] 20210616 ⚠  parse failed
[15/32] 20210922 ⚠  parse failed
[16/32] 20211215 ⚠  parse failed
[17/32] 20220316 ⚠  parse failed
[18/32] 20220615 ⚠  parse failed
[19/32] 20220921 ⚠  parse failed
[20/32] 20221214 ⚠  parse failed
[21/32] 20230322 ⚠  parse failed
[22/32] 20230614 ⚠  parse failed
[23/32] 20230920 ⚠  parse failed
[24/32] 20231213 ⚠  parse failed
[25/32] 20240320 ⚠  parse failed
[26/32] 20240612 ⚠  parse failed
[27/32] 20240918 ⚠  parse failed
[28/32] 20241218 ⚠  parse failed
[29/32] 20250319 ⚠  parse failed
[30/32] 20250618 ⚠  parse failed
[31/